In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from typing import Tuple

In [3]:
def perform_kmeans_clustering(X: np.ndarray, n_clusters: int = 3, random_state: int = 42) -> Tuple[KMeans, np.ndarray]:
    if not isinstance(X, np.ndarray):
        X = np.array(X)

    if X.ndim != 2:
        raise ValueError("X must be a 2D array of shape (n_samples, n_features)")

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X)
    centroids = kmeans.cluster_centers_
    return kmeans, centroids

In [4]:
file_path = "../data/processed/synthetic.csv"

data = pd.read_csv(filepath_or_buffer=file_path)
data.head()

,Unnamed: 0,username,public_repos,forked_repo_ratio,commit_count_last_6m,stars_total,followers,pull_requests_merged,avg_stars_per_repo,following,pull_requests_opened,issues_opened,readme_presence_ratio,cluster_label
0,0,ethan03,15,0.485018,329,9,3,33,1.715062,0,18,0,0.001544,Cluster_2
1,1,jenniferrogers,6,0.085403,0,61,25,1,30.863189,0,0,32,0.001431,Cluster_1
2,2,rhonda11,19,0.972461,448,3,192,37,0.000000,81,73,13,0.001421,Cluster_1
3,3,jessica24,72,0.518010,1403,170,10,141,29.912658,0,0,15,0.008967,Cluster_1
4,4,jrojas,5,0.614186,134,28,48,13,0.000000,6,0,0,0.000221,Cluster_3


In [14]:
data.columns

Index(['Unnamed: 0', 'username', 'public_repos', 'forked_repo_ratio',
       'commit_count_last_6m', 'stars_total', 'followers',
       'pull_requests_merged', 'avg_stars_per_repo', 'following',
       'pull_requests_opened', 'issues_opened', 'readme_presence_ratio',
       'cluster_label'],
      dtype='object')

In [15]:
selected_features = ['public_repos', 'forked_repo_ratio',
       'commit_count_last_6m', 'stars_total', 'followers',
       'pull_requests_merged', 'avg_stars_per_repo', 'following',
       'pull_requests_opened', 'issues_opened', 'readme_presence_ratio']

data_selected = data[selected_features]
data_selected.head()

,public_repos,forked_repo_ratio,commit_count_last_6m,stars_total,followers,pull_requests_merged,avg_stars_per_repo,following,pull_requests_opened,issues_opened,readme_presence_ratio
0,15,0.485018,329,9,3,33,1.715062,0,18,0,0.001544
1,6,0.085403,0,61,25,1,30.863189,0,0,32,0.001431
2,19,0.972461,448,3,192,37,0.000000,81,73,13,0.001421
3,72,0.518010,1403,170,10,141,29.912658,0,0,15,0.008967
4,5,0.614186,134,28,48,13,0.000000,6,0,0,0.000221


In [21]:
data_selected.replace([np.inf, -np.inf], np.nan, inplace=True)

/tmp/ipykernel_4827/2458466670.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_selected.replace([np.inf, -np.inf], np.nan, inplace=True)


In [22]:
np.isinf(data_selected).sum()

public_repos             0
forked_repo_ratio        0
commit_count_last_6m     0
stars_total              0
followers                0
pull_requests_merged     0
avg_stars_per_repo       0
following                0
pull_requests_opened     0
issues_opened            0
readme_presence_ratio    0
dtype: int64

In [24]:
from sklearn.preprocessing import MinMaxScaler

scale_min_max = MinMaxScaler(feature_range=(1,10))
data_scaled = scale_min_max.fit_transform(data_selected.values)
data_scaled

array([[1.0505997 , 5.36640005, 1.05550245, ..., 1.11885547, 1.        ,
        1.02048581],
       [1.02023988, 1.76705678, 1.        , ..., 1.        , 1.35820896,
        1.01898599],
       [1.06409295, 9.75681662, 1.0755778 , ..., 1.48202494, 1.14552239,
        1.01884501],
       ...,
       [1.01349325, 2.18504904, 1.02479897, ..., 1.        , 1.30223881,
        1.0038393 ],
       [1.29010495, 8.98336693, 1.30011809, ..., 1.1782832 , 1.45895522,
        1.12659451],
       [1.04047976, 7.76823027, 1.04791093, ..., 1.25752018, 1.62686567,
        1.0060938 ]], shape=(4000, 11))

In [25]:
np.isnan(data_scaled).sum()

np.int64(349)

In [26]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

data_final = imputer.fit_transform(data_scaled)

data_final

array([[1.0505997 , 5.36640005, 1.05550245, ..., 1.11885547, 1.        ,
        1.02048581],
       [1.02023988, 1.76705678, 1.        , ..., 1.        , 1.35820896,
        1.01898599],
       [1.06409295, 9.75681662, 1.0755778 , ..., 1.48202494, 1.14552239,
        1.01884501],
       ...,
       [1.01349325, 2.18504904, 1.02479897, ..., 1.        , 1.30223881,
        1.0038393 ],
       [1.29010495, 8.98336693, 1.30011809, ..., 1.1782832 , 1.45895522,
        1.12659451],
       [1.04047976, 7.76823027, 1.04791093, ..., 1.25752018, 1.62686567,
        1.0060938 ]], shape=(4000, 11))

In [27]:
model, centroids = perform_kmeans_clustering(data_final, n_clusters=5)
centroids

array([[1.08416349, 6.4816474 , 1.08459159, 1.07854276, 1.0346786 ,
        1.08425215, 1.03938317, 1.04934358, 1.13747138, 1.21079842,
        1.03344961],
       [1.03137181, 6.65318216, 1.03336895, 7.24441293, 1.03102141,
        1.03017984, 3.6596465 , 1.04728221, 1.04159941, 1.07164179,
        1.01883322],
       [1.08993482, 2.09923797, 1.09027595, 1.09095479, 1.03662946,
        1.0903117 , 1.03650025, 1.04915273, 1.14765334, 1.22189204,
        1.0408807 ],
       [1.06866336, 8.85049988, 1.06946998, 1.09198249, 1.03942511,
        1.06944174, 1.0394515 , 1.05556941, 1.11909927, 1.18867968,
        1.02722618],
       [1.07322777, 4.30397891, 1.07385148, 1.08051552, 1.03393005,
        1.0735004 , 1.03569843, 1.04621269, 1.13023069, 1.19672654,
        1.03009362]])

In [28]:
centroids.shape

(5, 11)

In [31]:
def assign_clusters(model: KMeans, X: np.ndarray) -> np.ndarray:
    if not hasattr(model, "predict"):
        raise ValueError("model must be a fitted KMeans instance")

    X = np.array(X)
    if X.ndim == 1:
        X = X.reshape(1, -1)

    return model.predict(X)

In [32]:
cluster_assignment = assign_clusters(model=model, X=data_final)

data_selected['clusters'] = cluster_assignment

/tmp/ipykernel_4827/956087623.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_selected['clusters'] = cluster_assignment


In [34]:
cluster_assignment

array([4, 2, 3, ..., 2, 3, 3], shape=(4000,), dtype=int32)

In [33]:
data_selected.head()

,public_repos,forked_repo_ratio,commit_count_last_6m,stars_total,followers,pull_requests_merged,avg_stars_per_repo,following,pull_requests_opened,issues_opened,readme_presence_ratio,clusters
0,15,0.485018,329,9,3,33,1.715062,0,18,0,0.001544,4
1,6,0.085403,0,61,25,1,30.863189,0,0,32,0.001431,2
2,19,0.972461,448,3,192,37,0.000000,81,73,13,0.001421,3
3,72,0.518010,1403,170,10,141,29.912658,0,0,15,0.008967,0
4,5,0.614186,134,28,48,13,0.000000,6,0,0,0.000221,0


In [35]:
centroids

array([[1.08416349, 6.4816474 , 1.08459159, 1.07854276, 1.0346786 ,
        1.08425215, 1.03938317, 1.04934358, 1.13747138, 1.21079842,
        1.03344961],
       [1.03137181, 6.65318216, 1.03336895, 7.24441293, 1.03102141,
        1.03017984, 3.6596465 , 1.04728221, 1.04159941, 1.07164179,
        1.01883322],
       [1.08993482, 2.09923797, 1.09027595, 1.09095479, 1.03662946,
        1.0903117 , 1.03650025, 1.04915273, 1.14765334, 1.22189204,
        1.0408807 ],
       [1.06866336, 8.85049988, 1.06946998, 1.09198249, 1.03942511,
        1.06944174, 1.0394515 , 1.05556941, 1.11909927, 1.18867968,
        1.02722618],
       [1.07322777, 4.30397891, 1.07385148, 1.08051552, 1.03393005,
        1.0735004 , 1.03569843, 1.04621269, 1.13023069, 1.19672654,
        1.03009362]])

In [37]:
cluster_summary = data_selected.groupby('clusters').describe()

In [43]:
cluster_summary[["public_repos", "forked_repo_ratio"]]

public_repos                         ... forked_repo_ratio                    
                count       mean         std  ...               50%       75%       max
clusters                                      ...                                      
0               995.0  24.845226   91.294261  ...          0.606512  0.670756  0.740510
1                10.0   9.300000    8.512083  ...          0.655340  0.858694  0.971845
2              1002.0  26.660679  102.037968  ...          0.122982  0.183549  0.244745
3               973.0  20.384378   48.852472  ...          0.871546  0.936177  0.999461
4              1020.0  21.782353   54.945005  ...          0.367979  0.430323  0.487900

[5 rows x 16 columns]

In [ ]:
selected_features = ['public_repos', 'forked_repo_ratio',
       'commit_count_last_6m', 'stars_total', 'followers',
       'pull_requests_merged', 'avg_stars_per_repo', 'following',
       'pull_requests_opened', 'issues_opened', 'readme_presence_ratio']

In [44]:
cluster_summary[['commit_count_last_6m', 'stars_total']]

commit_count_last_6m              ... stars_total         
                        count        mean  ...         75%      max
clusters                                   ...                     
0                       995.0  499.369849  ...       97.00   4558.0
1                        10.0  197.800000  ...    12295.75  15867.0
2                      1002.0  535.125749  ...      120.00   6940.0
3                       973.0  412.369990  ...      110.00   5522.0
4                      1020.0  439.240196  ...      118.50   4198.0

[5 rows x 16 columns]

In [46]:
cluster_summary[['followers',
       'pull_requests_merged', 'following']]

followers                               ... following                     
             count       mean         std   min  ...       25%   50%    75%     max
clusters                                         ...                               
0            995.0  63.961809  146.657667   0.0  ...       0.0  22.0  62.00  1089.0
1             10.0  57.300000   65.294291  11.0  ...       8.0  41.5  73.75   106.0
2           1002.0  67.658683  266.740639   0.0  ...       0.0  17.0  58.00  3719.0
3            973.0  72.947585  547.140530   0.0  ...       0.0  20.0  61.00  8242.0
4           1020.0  62.665686  141.756107   0.0  ...       0.0  15.5  57.00   811.0

[5 rows x 24 columns]

In [47]:
cluster_summary[['pull_requests_opened', 'issues_opened']]

pull_requests_opened                        ... issues_opened              
                        count       mean        std  ...           50%    75%    max
clusters                                             ...                            
0                       995.0  20.787940  50.339954  ...           9.0  29.50  722.0
1                        10.0   6.300000  12.046669  ...           0.0   7.25   29.0
2                      1002.0  22.361277  54.136664  ...          11.0  30.00  804.0
3                       973.0  18.028777  29.372540  ...           8.0  27.00  229.0
4                      1020.0  19.758824  33.300792  ...           8.0  30.00  263.0

[5 rows x 16 columns]

In [48]:
cluster_summary[['readme_presence_ratio', 'avg_stars_per_repo' ]]

readme_presence_ratio            ... avg_stars_per_repo             
                         count      mean  ...                75%          max
clusters                                  ...                                
0                        995.0  0.002507  ...          30.579384  3252.210282
1                         10.0  0.001420  ...        5291.346470  9417.639766
2                       1002.0  0.003082  ...          32.514362  1987.659137
3                        973.0  0.002056  ...          32.170651  2699.830555
4                       1020.0  0.002279  ...          30.364104  2099.498883

[5 rows x 16 columns]

In [55]:
test_data = data_final[1:6, :]
test_data.shape

(5, 11)

In [56]:
prediction = model.fit_predict(test_data)
prediction

array([3, 0, 2, 4, 1], dtype=int32)

In [57]:
cluster_assignment

array([4, 2, 3, ..., 2, 3, 3], shape=(4000,), dtype=int32)

In [50]:
data_final.shape

(4000, 11)

# Final Answer

The following clusters represent the results based on the score bands:

- **Cluster 0:** 5 / 100 (0–20 band)
- **Cluster 3:** 52 / 100 (40–60 band)
- **Cluster 4:** 58 / 100 (40–60 band)
- **Cluster 2:** 72 / 100 (60–80 band)
- **Cluster 1:** 95 / 100 (80–100 band)


### Summary of GitHub User Clusters

#### Cluster 0: Large-Scale Inactive or Dormant Users
- **Size**: Very large (995 users).
- **Key Traits**: Extremely low activity across the board—near-zero commits in the last 6 months (mean: 499, but skewed by outliers; most have 0), minimal stars (mean: 139 total, median: 30), few followers (mean: 64, median: 20), and almost no open issues/PRs (medians: 0). Public repos are abundant (mean: 24.8k, but median: 2, indicating many empty accounts), with low fork ratios (~0.61 mean). README presence is rare (mean ratio: 0.0025).
- **Interpretation**: Likely abandoned or "ghost" accounts, spammers, or casual sign-ups with high repo counts but no engagement. Low avg stars per repo (~44, but median: 10) suggests little impact.

#### Cluster 1: Small Elite Active Contributors
- **Size**: Tiny (10 users).
- **Key Traits**: High engagement in a compact footprint—few repos (mean: 9.3, median: 9.5) but explosive activity (commits last 6m: mean 198, up to 572; stars: mean 1,100 total, up to 15k+). Strong social pull (followers: mean 57, median 35; following: mean 45). Pull requests merged (mean: 17.9, median: 9) and open issues (mean: 6.4, median: 0) indicate focused maintenance. Fork ratio is low (~0.63), and avg stars per repo is sky-high (~3,091, median: 549).
- **Interpretation**: Top-tier individual developers or influencers (e.g., core maintainers of popular solo projects). High productivity per repo, but limited scale—think niche open-source stars.

#### Cluster 2: Mid-Tier Balanced Organizations
- **Size**: Medium-large (1,002 users).
- **Key Traits**: Moderate repos (mean: 26.7, median: 7) with steady activity (commits last 6m: mean 535, median: 151; stars: mean 160 total, median: 35). Followers (mean: 68, median: 21) and following (mean: 45, median: 17) suggest collaborative networks. Issues/PRs show consistent workflow (open issues: mean 19, median 11; merged PRs: mean 54, median: 16). Fork ratio (~1.12 mean) indicates some community interest; avg stars per repo (~41, median: 49) is solid but not elite.
- **Interpretation**: Active teams or mid-sized orgs (e.g., small companies or hobby groups) with reliable, distributed contributions. Balanced growth without extremes.

#### Cluster 3: Steady Volume Producers
- **Size**: Medium (973 users).
- **Key Traits**: Similar to Cluster 2 in repo scale (mean: 20.4, median: 6) and activity (commits last 6m: mean 412, median: 139; stars: mean 162 total, median: 33). Slightly higher followers (mean: 72, median: 18) and following (mean: 51, median: 20). PRs merged (mean: 41, median: 14) and open issues (mean: 17, median: 8) point to ongoing maintenance. Fork ratio (~0.87 mean) is moderate; avg stars per repo (~44, median: 48).
- **Interpretation**: Reliable "workhorse" users or orgs (e.g., internal teams or consistent contributors) focused on volume over virality. Overlaps with Cluster 2 but with marginally higher social reach.

#### Cluster 4: High-Volume High-Engagement Powerhouses
- **Size**: Medium (1,020 users).
- **Key Traits**: Large repo portfolios (mean: 21.8, median: 8) paired with robust activity (commits last 6m: mean 439, median: 160; stars: mean 142 total, median: 36). Strong networks (followers: mean 63, median: 20; following: mean 42, median: 15). Elevated PRs (merged: mean 44, median: 17) and issues (open: mean 18, median: 8). Higher fork ratio (~0.37 mean, but wider variance) suggests broader collaboration; avg stars per repo (~39, median: 30) indicates quality scale.
- **Interpretation**: Professional orgs or prolific individuals (e.g., enterprise teams or serial creators) with sustained, high-output ecosystems. More forked/collaborative than Clusters 2/3, emphasizing impact at scale.
